# Time-Series Momentum

Compares momentum signals and volatility estimators on SPY against a buy-and-hold URTH benchmark.

Daily OHLCV comes from Yahoo Finance via yfinance. The backtest uses split-adjusted prices and excludes dividend adjustment.

In [ ]:
import pandas as pd

from _market_data import download_ohlcv
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

class TradingConfig:
    """
    Centralized configuration for all trading parameters.
    All parameters are configurable and avoid hardcoding.
    """
    
    
    # Backtest Period
    END_DATE = "2024-12-31"
    YEARS_BACK = 5
    START_DATE = (datetime.strptime(END_DATE, "%Y-%m-%d") - 
                  timedelta(days=365 * YEARS_BACK)).strftime("%Y-%m-%d")
    
    # Trading Parameters
    COMMISSION_RATE = 0.001  # 0.1% per trade (realistic commission)
    INITIAL_CAPITAL = 100000  # Starting capital
    
    # Benchmark
    BENCHMARK_TICKER = "URTH"  # Default benchmark (can be changed)
    
    # Momentum Indicator Configurations
    # We'll test different types of momentum indicators
    MOMENTUM_TYPES = ["price_momentum", "sma_crossover", "sma_slope"]
    
    # SMA windows for momentum calculation
    SMA_FAST_WINDOWS = [20, 50]  # Fast SMA periods
    SMA_SLOW_WINDOWS = [50, 100, 200]  # Slow SMA periods
    
    # Price momentum windows (original TSMOM)
    PRICE_MOMENTUM_WINDOWS = [20, 60, 120]  # Lookback periods for price momentum
    
    # Volatility Estimator Types
    VOLATILITY_TYPES = ["rolling", "parkinson", "garman_klass", "yang_zhang"]
    
    # Volatility calculation windows
    VOLATILITY_WINDOWS = [20, 40, 60]  # Different volatility estimation windows
    
    # Target Volatility for position sizing
    TARGET_VOLATILITY = 0.15  # 15% annualized volatility target
    
    # Universe of global indices for trading
    GLOBAL_INDICES = [
        "SPY",   # S&P 500
        "EFA",   # MSCI EAFE (Developed Markets ex US)
        "EEM",   # Emerging Markets
        "AGG",   # US Aggregate Bonds
        "GLD",   # Gold
        "DBC",   # Commodities
        "VNQ",   # Real Estate
        "TLT",   # Long-term Treasury
    ]
    
    # Trading instrument (which ticker to trade)
    TRADING_TICKER = "SPY"  # Can be changed to any ticker in GLOBAL_INDICES
    
    # Plot Configuration
    PLOT_FIGSIZE = (16, 10)
    PLOT_DPI = 100


# ============================================================================
# MOMENTUM INDICATORS
# ============================================================================

class MomentumIndicators:
    """
    Calculate various momentum indicators for signal generation.
    All calculations strictly avoid look-ahead bias using shift(1).
    """
    
    @staticmethod
    def calculate_price_momentum(close_prices: pd.Series, window: int) -> pd.Series:
        """
        Calculate price momentum (simple return over lookback period).
        
        Momentum[t] = (Close[t-1] / Close[t-1-window]) - 1
        Signal generated at t uses data up to t-1 only (NO LOOK-AHEAD BIAS).
        
        Args:
            close_prices: Series of closing prices
            window: Lookback period in days
            
        Returns:
            Series of momentum signals (+1 for long, -1 for short, 0 for neutral)
        """
        # Calculate return over window, shifted to avoid look-ahead
        momentum_return = close_prices.pct_change(window).shift(1)
        
        # Generate signal: +1 if positive momentum, -1 if negative
        signal = np.sign(momentum_return)
        
        return signal
    
    @staticmethod
    def calculate_sma(prices: pd.Series, window: int) -> pd.Series:
        """
        Calculate Simple Moving Average with proper time-shifting.
        
        SMA[t] uses prices from t-window to t-1 (NO LOOK-AHEAD BIAS).
        
        Args:
            prices: Series of prices
            window: Window for SMA calculation
            
        Returns:
            Series of SMA values
        """
        # Calculate SMA and shift to ensure we only use past data
        sma = prices.rolling(window=window).mean().shift(1)
        return sma
    
    @staticmethod
    def calculate_sma_crossover(close_prices: pd.Series, 
                                fast_window: int, 
                                slow_window: int) -> pd.Series:
        """
        Calculate SMA crossover signal.
        
        Signal logic:
        - Long (+1) when Fast SMA > Slow SMA (golden cross)
        - Short (-1) when Fast SMA < Slow SMA (death cross)
        
        Both SMAs calculated at t-1 to avoid look-ahead bias.
        
        Args:
            close_prices: Series of closing prices
            fast_window: Window for fast SMA
            slow_window: Window for slow SMA
            
        Returns:
            Series of signals (+1, -1, or 0)
        """
        # Calculate both SMAs (already shifted in calculate_sma)
        fast_sma = MomentumIndicators.calculate_sma(close_prices, fast_window)
        slow_sma = MomentumIndicators.calculate_sma(close_prices, slow_window)
        
        # Generate crossover signal
        signal = pd.Series(0, index=close_prices.index)
        signal[fast_sma > slow_sma] = 1   # Golden cross - go long
        signal[fast_sma < slow_sma] = -1  # Death cross - go short
        
        return signal
    
    @staticmethod
    def calculate_sma_slope(close_prices: pd.Series, 
                           sma_window: int, 
                           slope_window: int = 5) -> pd.Series:
        """
        Calculate signal based on SMA slope (rate of change).
        
        Signal logic:
        - Long (+1) when SMA is trending upward
        - Short (-1) when SMA is trending downward
        
        SMA slope = (SMA[t-1] - SMA[t-1-slope_window]) / slope_window
        
        Args:
            close_prices: Series of closing prices
            sma_window: Window for SMA calculation
            slope_window: Window for slope calculation
            
        Returns:
            Series of signals (+1, -1, or 0)
        """
        # Calculate SMA (already shifted)
        sma = MomentumIndicators.calculate_sma(close_prices, sma_window)
        
        # Calculate SMA slope (change over slope_window)
        sma_slope = sma.diff(slope_window) / slope_window
        
        # Generate signal based on slope direction
        signal = np.sign(sma_slope)
        
        return signal


# ============================================================================
# VOLATILITY ESTIMATORS
# ============================================================================

class VolatilityEstimators:
    """
    Advanced volatility estimators using OHLC data.
    These estimators are more efficient than simple close-to-close volatility
    because they use intraday range information.
    
    All estimators are calculated with proper time-shifting to avoid look-ahead bias.
    """
    
    @staticmethod
    def calculate_rolling_volatility(close_prices: pd.Series, window: int) -> pd.Series:
        """
        Classic rolling volatility using close-to-close returns.
        
        This is the simplest volatility estimator based on standard deviation
        of logarithmic returns.
        
        Vol[t] = std(returns from t-window to t-1) * sqrt(252)
        
        Args:
            close_prices: Series of closing prices
            window: Rolling window for volatility calculation
            
        Returns:
            Series of annualized volatility
        """
        # Calculate log returns
        log_returns = np.log(close_prices / close_prices.shift(1))
        
        # Calculate rolling standard deviation (shifted to avoid look-ahead)
        rolling_std = log_returns.rolling(window=window).std().shift(1)
        
        # Annualize volatility (252 trading days per year)
        annualized_vol = rolling_std * np.sqrt(252)
        
        return annualized_vol
    
    @staticmethod
    def calculate_parkinson_volatility(high_prices: pd.Series, 
                                       low_prices: pd.Series, 
                                       window: int) -> pd.Series:
        """
        Parkinson volatility estimator (1980).
        
        Uses High-Low range to estimate volatility. More efficient than 
        close-to-close because it captures intraday volatility.
        
        Formula: Vol = sqrt((1/(4*ln(2))) * mean(ln(High/Low)^2))
        
        Advantage: 5x more efficient than close-to-close estimator.
        Limitation: Assumes no drift (zero mean return).
        
        Args:
            high_prices: Series of daily high prices
            low_prices: Series of daily low prices
            window: Rolling window for calculation
            
        Returns:
            Series of annualized Parkinson volatility
        """
        # Calculate log(High/Low) ratio
        hl_ratio = np.log(high_prices / low_prices)
        
        # Parkinson variance: (1/(4*ln(2))) * mean(ln(H/L)^2)
        parkinson_var = (1.0 / (4.0 * np.log(2))) * (hl_ratio ** 2)
        
        # Calculate rolling mean variance (shifted to avoid look-ahead)
        rolling_var = parkinson_var.rolling(window=window).mean().shift(1)
        
        # Convert to volatility (sqrt of variance) and annualize
        parkinson_vol = np.sqrt(rolling_var) * np.sqrt(252)
        
        return parkinson_vol
    
    @staticmethod
    def calculate_garman_klass_volatility(open_prices: pd.Series,
                                         high_prices: pd.Series,
                                         low_prices: pd.Series,
                                         close_prices: pd.Series,
                                         window: int) -> pd.Series:
        """
        Garman-Klass volatility estimator (1980).
        
        Uses OHLC data to estimate volatility. More efficient than Parkinson
        because it also incorporates opening and closing price information.
        
        Formula: Vol = sqrt(0.5*(ln(H/L))^2 - (2*ln(2)-1)*(ln(C/O))^2)
        
        Advantage: 7.4x more efficient than close-to-close estimator.
        Limitation: Assumes zero drift and no opening gaps.
        
        Args:
            open_prices: Series of daily opening prices
            high_prices: Series of daily high prices
            low_prices: Series of daily low prices
            close_prices: Series of daily closing prices
            window: Rolling window for calculation
            
        Returns:
            Series of annualized Garman-Klass volatility
        """
        # Calculate log ratios
        hl_ratio = np.log(high_prices / low_prices)
        co_ratio = np.log(close_prices / open_prices)
        
        # Garman-Klass variance
        # 0.5 * (ln(H/L))^2 - (2*ln(2)-1) * (ln(C/O))^2
        gk_var = 0.5 * (hl_ratio ** 2) - (2 * np.log(2) - 1) * (co_ratio ** 2)
        
        # Calculate rolling mean variance (shifted to avoid look-ahead)
        rolling_var = gk_var.rolling(window=window).mean().shift(1)
        
        # Convert to volatility and annualize
        gk_vol = np.sqrt(rolling_var) * np.sqrt(252)
        
        return gk_vol
    
    @staticmethod
    def calculate_yang_zhang_volatility(open_prices: pd.Series,
                                       high_prices: pd.Series,
                                       low_prices: pd.Series,
                                       close_prices: pd.Series,
                                       window: int) -> pd.Series:
        """
        Yang-Zhang volatility estimator (2000).
        
        The most sophisticated OHLC volatility estimator. Combines:
        - Overnight volatility (close-to-open)
        - Opening jump volatility
        - Intraday drift-independent volatility (Rogers-Satchell)
        
        Advantage: Handles both drift and opening gaps, most robust estimator.
        Advantage: 14x more efficient than close-to-close estimator.
        
        Formula: Vol = sqrt(Vol_overnight + k*Vol_open + (1-k)*Vol_rs)
        where k is a weighting factor based on window size.
        
        Args:
            open_prices: Series of daily opening prices
            high_prices: Series of daily high prices
            low_prices: Series of daily low prices
            close_prices: Series of daily closing prices
            window: Rolling window for calculation
            
        Returns:
            Series of annualized Yang-Zhang volatility
        """
        # Calculate log ratios
        # Overnight: Close[t-1] to Open[t]
        overnight_ret = np.log(open_prices / close_prices.shift(1))
        
        # Open-to-close return
        oc_ret = np.log(close_prices / open_prices)
        
        # Rogers-Satchell volatility components
        # Uses High, Low, Open, Close to capture intraday volatility without drift
        rs1 = np.log(high_prices / close_prices) * np.log(high_prices / open_prices)
        rs2 = np.log(low_prices / close_prices) * np.log(low_prices / open_prices)
        
        # Calculate variance components
        # Overnight variance
        overnight_var = overnight_ret.rolling(window=window).var()
        
        # Open variance
        open_var = oc_ret.rolling(window=window).var()
        
        # Rogers-Satchell variance
        rs_var = (rs1 + rs2).rolling(window=window).mean()
        
        # Calculate weighting factor k based on window size
        # k = 0.34 / (1.34 + (window+1)/(window-1))
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        
        # Yang-Zhang variance: overnight + k*open + (1-k)*rogers_satchell
        yz_var = overnight_var + k * open_var + (1 - k) * rs_var
        
        # Shift to avoid look-ahead bias
        yz_var = yz_var.shift(1)
        
        # Convert to volatility and annualize
        yz_vol = np.sqrt(yz_var) * np.sqrt(252)
        
        return yz_vol


# ============================================================================
# TSMOM STRATEGY ENGINE WITH FLEXIBLE INDICATORS
# ============================================================================

class TSMOMStrategy:
    """
    Time Series Momentum (TSMOM) Strategy with flexible momentum and volatility indicators.
    
    Strategy Logic:
    1. Calculate momentum signal using chosen indicator (price momentum, SMA, etc.)
    2. Calculate realized volatility using chosen estimator (rolling, Parkinson, GK, YZ)
    3. Signal generated at Close[t]
    4. Execution at Open[t+1]
    5. Portfolio valued at Open[t+1]
    6. Returns = Open[t+2]/Open[t+1] - 1
    
    NO LOOK-AHEAD BIAS: All calculations use only data available at time t-1
    """
    
    def __init__(self, 
                 momentum_type: str,
                 momentum_params: Dict,
                 volatility_type: str,
                 volatility_window: int,
                 target_volatility: float,
                 commission_rate: float):
        """
        Initialize TSMOM strategy with flexible indicators.
        
        Args:
            momentum_type: Type of momentum indicator 
                          ('price_momentum', 'sma_crossover', 'sma_slope')
            momentum_params: Dictionary of parameters for momentum calculation
            volatility_type: Type of volatility estimator
                           ('rolling', 'parkinson', 'garman_klass', 'yang_zhang')
            volatility_window: Window for volatility calculation
            target_volatility: Target portfolio volatility for position sizing
            commission_rate: Commission per trade (e.g., 0.001 = 0.1%)
        """
        self.momentum_type = momentum_type
        self.momentum_params = momentum_params
        self.volatility_type = volatility_type
        self.volatility_window = volatility_window
        self.target_volatility = target_volatility
        self.commission_rate = commission_rate
    
    def calculate_momentum_signal(self, ohlc_data: pd.DataFrame) -> pd.Series:
        """
        Calculate momentum signal using specified indicator.
        
        Args:
            ohlc_data: DataFrame with OHLC price data
            
        Returns:
            Series of momentum signals (+1 for long, -1 for short, 0 for neutral)
        """
        close_prices = ohlc_data["close"]
        
        if self.momentum_type == "price_momentum":
            # Simple price momentum over lookback window
            window = self.momentum_params.get("window", 60)
            signal = MomentumIndicators.calculate_price_momentum(close_prices, window)
            
        elif self.momentum_type == "sma_crossover":
            # SMA crossover strategy (fast vs slow)
            fast_window = self.momentum_params.get("fast_window", 20)
            slow_window = self.momentum_params.get("slow_window", 50)
            signal = MomentumIndicators.calculate_sma_crossover(
                close_prices, fast_window, slow_window
            )
            
        elif self.momentum_type == "sma_slope":
            # SMA slope direction
            sma_window = self.momentum_params.get("sma_window", 50)
            slope_window = self.momentum_params.get("slope_window", 5)
            signal = MomentumIndicators.calculate_sma_slope(
                close_prices, sma_window, slope_window
            )
        else:
            raise ValueError(f"Unknown momentum type: {self.momentum_type}")
        
        return signal
    
    def calculate_realized_volatility(self, ohlc_data: pd.DataFrame) -> pd.Series:
        """
        Calculate realized volatility using specified estimator.
        
        Args:
            ohlc_data: DataFrame with OHLC price data
            
        Returns:
            Series of realized volatility (annualized)
        """
        if self.volatility_type == "rolling":
            # Classic rolling volatility using close prices
            volatility = VolatilityEstimators.calculate_rolling_volatility(
                ohlc_data["close"], self.volatility_window
            )
            
        elif self.volatility_type == "parkinson":
            # Parkinson volatility (High-Low)
            volatility = VolatilityEstimators.calculate_parkinson_volatility(
                ohlc_data["high"], ohlc_data["low"], self.volatility_window
            )
            
        elif self.volatility_type == "garman_klass":
            # Garman-Klass volatility (OHLC)
            volatility = VolatilityEstimators.calculate_garman_klass_volatility(
                ohlc_data["open"], ohlc_data["high"], 
                ohlc_data["low"], ohlc_data["close"], 
                self.volatility_window
            )
            
        elif self.volatility_type == "yang_zhang":
            # Yang-Zhang volatility (OHLC with overnight)
            volatility = VolatilityEstimators.calculate_yang_zhang_volatility(
                ohlc_data["open"], ohlc_data["high"], 
                ohlc_data["low"], ohlc_data["close"], 
                self.volatility_window
            )
        else:
            raise ValueError(f"Unknown volatility type: {self.volatility_type}")
        
        return volatility
    
    def calculate_position_size(self, signal: pd.Series, realized_vol: pd.Series) -> pd.Series:
        """
        Calculate position size based on volatility targeting.
        
        Position Size = Signal * (Target Vol / Realized Vol)
        
        This scales positions inversely with volatility to maintain constant risk.
        
        Args:
            signal: Trading signal (+1, -1, or 0)
            realized_vol: Realized volatility
            
        Returns:
            Series of position sizes (fraction of capital)
        """
        # Avoid division by zero or very small volatility
        realized_vol_safe = realized_vol.replace(0, np.nan)
        
        # Calculate target position size
        position_size = signal * (self.target_volatility / realized_vol_safe)
        
        # Cap position size at 1.0 (100% of capital) for safety
        # Also cap at -1.0 for short positions
        position_size = position_size.clip(-1.0, 1.0)
        
        return position_size
    
    def simulate_trading(self, ohlc_data: pd.DataFrame) -> pd.DataFrame:
        """
        Simulate trading strategy with realistic execution.
        
        Trading Flow:
        - Signal generated at Close[t]
        - Execute at Open[t+1]
        - Portfolio valued at Open[t+1]
        - Returns = Open[t+2]/Open[t+1] - 1
        
        Args:
            ohlc_data: DataFrame with OHLC price data
            
        Returns:
            DataFrame with signals, positions, and returns
        """
        results = pd.DataFrame(index=ohlc_data.index)
        
        # Step 1: Calculate momentum signal
        results["momentum_signal"] = self.calculate_momentum_signal(ohlc_data)
        
        # Step 2: Calculate realized volatility
        results["realized_volatility"] = self.calculate_realized_volatility(ohlc_data)
        
        # Step 3: Calculate position size based on vol targeting
        results["position_size"] = self.calculate_position_size(
            results["momentum_signal"], 
            results["realized_volatility"]
        )
        
        # Step 4: Calculate execution prices and returns
        results["open"] = ohlc_data["open"]
        results["close"] = ohlc_data["close"]
        
        # Execution: Position decided at Close[t] is executed at Open[t+1]
        results["executed_position"] = results["position_size"].shift(1)
        
        # Returns during trading: Open[t+1] to Open[t+2]
        results["open_to_open_return"] = ohlc_data["open"].pct_change()
        
        # Strategy returns (before commissions)
        results["gross_return"] = results["executed_position"] * results["open_to_open_return"]
        
        # Calculate commissions (charged on position changes)
        results["position_change"] = results["executed_position"].diff().abs()
        results["commission_cost"] = results["position_change"] * self.commission_rate
        
        # Net returns (after commissions)
        results["net_return"] = results["gross_return"] - results["commission_cost"]
        
        # During holding period: Close to Close returns (for reference)
        results["close_to_close_return"] = ohlc_data["close"].pct_change()
        
        return results


# ============================================================================
# BENCHMARK CALCULATION
# ============================================================================

class BenchmarkCalculator:
    """
    Calculate benchmark returns following buy-and-hold strategy.
    
    For benchmark, we maintain Close-to-Close returns since there are no operations.
    """
    
    def __init__(self, commission_rate: float):
        self.commission_rate = commission_rate
    
    def calculate_benchmark_returns(self, ohlc_data: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate benchmark returns (buy and hold).
        
        Args:
            ohlc_data: DataFrame with OHLC data
            
        Returns:
            DataFrame with benchmark returns
        """
        results = pd.DataFrame(index=ohlc_data.index)
        
        # For benchmark: Close to Close returns (no operations during period)
        results["benchmark_return"] = ohlc_data["close"].pct_change()
        
        # First period: buy at first open, so we pay commission once
        results["commission_cost"] = 0.0
        if len(results) > 0:
            results.loc[results.index[0], "commission_cost"] = self.commission_rate
        
        # Net benchmark return
        results["net_benchmark_return"] = results["benchmark_return"] - results["commission_cost"]
        
        return results


# ============================================================================
# PERFORMANCE METRICS
# ============================================================================

class PerformanceAnalyzer:
    """
    Calculate comprehensive performance metrics for strategy evaluation.
    """
    
    @staticmethod
    def calculate_cumulative_returns(returns: pd.Series) -> pd.Series:
        """Calculate cumulative returns from period returns."""
        return (1 + returns).cumprod()
    
    @staticmethod
    def calculate_sharpe_ratio(returns: pd.Series, risk_free_rate: float = 0.02) -> float:
        """
        Calculate annualized Sharpe ratio.
        
        Args:
            returns: Series of period returns
            risk_free_rate: Annual risk-free rate (default 2%)
        """
        if len(returns) == 0 or returns.std() == 0:
            return 0.0
        
        excess_returns = returns - risk_free_rate / 252
        sharpe = np.sqrt(252) * excess_returns.mean() / returns.std()
        return sharpe
    
    @staticmethod
    def calculate_max_drawdown(cumulative_returns: pd.Series) -> float:
        """Calculate maximum drawdown."""
        running_max = cumulative_returns.expanding().max()
        drawdown = (cumulative_returns - running_max) / running_max
        return drawdown.min()
    
    @staticmethod
    def calculate_annual_return(cumulative_returns: pd.Series) -> float:
        """Calculate annualized return."""
        if len(cumulative_returns) == 0:
            return 0.0
        
        total_return = cumulative_returns.iloc[-1] / cumulative_returns.iloc[0] - 1
        years = len(cumulative_returns) / 252
        annual_return = (1 + total_return) ** (1 / years) - 1
        return annual_return
    
    @staticmethod
    def calculate_volatility(returns: pd.Series) -> float:
        """Calculate annualized volatility."""
        return returns.std() * np.sqrt(252)
    
    @staticmethod
    def calculate_metrics(returns: pd.Series) -> Dict[str, float]:
        """
        Calculate all performance metrics.
        
        Args:
            returns: Series of period returns
            
        Returns:
            Dictionary of performance metrics
        """
        cum_returns = PerformanceAnalyzer.calculate_cumulative_returns(returns)
        
        metrics = {
            "Total Return": cum_returns.iloc[-1] - 1 if len(cum_returns) > 0 else 0.0,
            "Annual Return": PerformanceAnalyzer.calculate_annual_return(cum_returns),
            "Volatility": PerformanceAnalyzer.calculate_volatility(returns),
            "Sharpe Ratio": PerformanceAnalyzer.calculate_sharpe_ratio(returns),
            "Max Drawdown": PerformanceAnalyzer.calculate_max_drawdown(cum_returns),
        }
        
        return metrics


# ============================================================================
# VISUALIZATION
# ============================================================================

class StrategyVisualizer:
    """
    Create visualizations for strategy performance comparison.
    All plot parameters are configurable (no hardcoding).
    """
    
    def __init__(self, figsize: Tuple[int, int], dpi: int):
        self.figsize = figsize
        self.dpi = dpi
    
    def plot_cumulative_returns(self, results_dict: Dict[str, pd.Series], 
                                benchmark_returns: pd.Series,
                                title: str = "Cumulative Returns Comparison",
                                max_strategies: int = 15):
        """
        Plot cumulative returns for multiple strategies vs benchmark.
        
        Args:
            results_dict: Dictionary mapping strategy name to return series
            benchmark_returns: Benchmark return series
            title: Plot title
            max_strategies: Maximum number of strategies to plot (for readability)
        """
        fig, ax = plt.subplots(figsize=self.figsize, dpi=self.dpi)
        
        # Plot benchmark first
        benchmark_cum = PerformanceAnalyzer.calculate_cumulative_returns(benchmark_returns)
        ax.plot(benchmark_cum.index, benchmark_cum.values, 
                label="Benchmark", linewidth=2.5, color="black", linestyle="--")
        
        # Select top strategies by Sharpe ratio for plotting
        if len(results_dict) > max_strategies:
            # Calculate Sharpe for all strategies
            sharpe_dict = {name: PerformanceAnalyzer.calculate_sharpe_ratio(returns) 
                          for name, returns in results_dict.items()}
            # Sort by Sharpe and take top N
            top_strategies = sorted(sharpe_dict.items(), 
                                   key=lambda x: x[1], 
                                   reverse=True)[:max_strategies]
            strategies_to_plot = {name: results_dict[name] for name, _ in top_strategies}
            print(f"\nPlotting top {max_strategies} strategies by Sharpe Ratio")
        else:
            strategies_to_plot = results_dict
        
        # Plot strategies
        colors = plt.cm.tab20(np.linspace(0, 1, len(strategies_to_plot)))
        for (strategy_name, returns), color in zip(strategies_to_plot.items(), colors):
            cum_returns = PerformanceAnalyzer.calculate_cumulative_returns(returns)
            ax.plot(cum_returns.index, cum_returns.values, 
                    label=strategy_name, linewidth=1.2, color=color, alpha=0.7)
        
        ax.set_xlabel("Date", fontsize=12)
        ax.set_ylabel("Cumulative Return", fontsize=12)
        ax.set_title(title, fontsize=14, fontweight="bold")
        ax.legend(loc="best", fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def plot_performance_heatmap(self, metrics_df: pd.DataFrame, 
                                title: str = "Performance Metrics Heatmap",
                                top_n: int = 20):
        """
        Create heatmap of performance metrics across strategies.
        
        Args:
            metrics_df: DataFrame with strategies as rows and metrics as columns
            title: Plot title
            top_n: Show only top N strategies by Sharpe Ratio
        """
        # Sort by Sharpe Ratio and take top N
        if "Sharpe Ratio" in metrics_df.columns and len(metrics_df) > top_n:
            metrics_df = metrics_df.nlargest(top_n, "Sharpe Ratio")
            title = f"{title} (Top {top_n} by Sharpe)"
        
        fig, ax = plt.subplots(figsize=self.figsize, dpi=self.dpi)
        
        sns.heatmap(metrics_df.T, annot=True, fmt=".3f", cmap="RdYlGn", 
                   center=0, ax=ax, cbar_kws={"label": "Value"})
        
        ax.set_title(title, fontsize=14, fontweight="bold")
        ax.set_xlabel("Strategy", fontsize=12)
        ax.set_ylabel("Metric", fontsize=12)
        plt.tight_layout()
        plt.show()
    
    def plot_volatility_comparison(self, results_dict: Dict[str, pd.DataFrame],
                                   title: str = "Volatility Estimator Comparison"):
        """
        Plot realized volatility over time for different estimators.
        
        Args:
            results_dict: Dictionary mapping strategy name to results DataFrame
            title: Plot title
        """
        fig, ax = plt.subplots(figsize=self.figsize, dpi=self.dpi)
        
        colors = plt.cm.Set2(np.linspace(0, 1, len(results_dict)))
        for (strategy_name, results), color in zip(results_dict.items(), colors):
            if "realized_volatility" in results.columns:
                vol = results["realized_volatility"].dropna()
                ax.plot(vol.index, vol.values, 
                       label=strategy_name, linewidth=1.5, color=color, alpha=0.7)
        
        ax.set_xlabel("Date", fontsize=12)
        ax.set_ylabel("Annualized Volatility", fontsize=12)
        ax.set_title(title, fontsize=14, fontweight="bold")
        ax.legend(loc="best", fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function to run TSMOM strategy backtests.
    
    This function:
    1. Downloads historical data from Yahoo Finance
    2. Runs multiple TSMOM configurations with different momentum/volatility combinations
    3. Compares results against benchmark
    4. Displays performance metrics and visualizations
    """
    
    # Initialize configuration
    config = TradingConfig()
    
    print("=" * 80)
    print("ENHANCED TSMOM STRATEGY BACKTEST FRAMEWORK")
    print("Multiple Momentum Indicators × Multiple Volatility Estimators")
    print("=" * 80)
    print(f"Period: {config.START_DATE} to {config.END_DATE}")
    print(f"Trading Instrument: {config.TRADING_TICKER}")
    print(f"Benchmark: {config.BENCHMARK_TICKER}")
    print(f"Commission Rate: {config.COMMISSION_RATE * 100:.2f}%")
    print(f"Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print("=" * 80)
    
    # Step 1: Download daily prices.
    print("\nDownloading data from Yahoo Finance...")
    trading_data = download_ohlcv(config.TRADING_TICKER, config.START_DATE, config.END_DATE)
    benchmark_data = download_ohlcv(config.BENCHMARK_TICKER, config.START_DATE, config.END_DATE)
    trading_data = trading_data.rename(columns=str.lower)
    benchmark_data = benchmark_data.rename(columns=str.lower)
    print(f"  {config.TRADING_TICKER}: {len(trading_data)} trading days")
    print(f"  {config.BENCHMARK_TICKER}: {len(benchmark_data)} trading days")

    # Step 2: Calculate benchmark returns
    print(f"\nCalculating benchmark returns for {config.BENCHMARK_TICKER}...")
    benchmark_calc = BenchmarkCalculator(config.COMMISSION_RATE)
    benchmark_results = benchmark_calc.calculate_benchmark_returns(benchmark_data)
    
    # Step 3: Generate strategy configurations
    print("\nGenerating strategy configurations...")
    strategy_configs = []
    
    # Price Momentum strategies
    for window in config.PRICE_MOMENTUM_WINDOWS:
        for vol_type in config.VOLATILITY_TYPES:
            for vol_window in config.VOLATILITY_WINDOWS:
                strategy_configs.append({
                    "momentum_type": "price_momentum",
                    "momentum_params": {"window": window},
                    "volatility_type": vol_type,
                    "volatility_window": vol_window,
                    "name": f"PriceMom{window}_{vol_type.upper()}{vol_window}"
                })
    
    # SMA Crossover strategies
    for fast_w in config.SMA_FAST_WINDOWS:
        for slow_w in config.SMA_SLOW_WINDOWS:
            if fast_w >= slow_w:
                continue  # Skip invalid combinations
            for vol_type in config.VOLATILITY_TYPES:
                for vol_window in config.VOLATILITY_WINDOWS:
                    strategy_configs.append({
                        "momentum_type": "sma_crossover",
                        "momentum_params": {"fast_window": fast_w, "slow_window": slow_w},
                        "volatility_type": vol_type,
                        "volatility_window": vol_window,
                        "name": f"SMACross{fast_w}_{slow_w}_{vol_type.upper()}{vol_window}"
                    })
    
    # SMA Slope strategies
    for sma_w in config.SMA_SLOW_WINDOWS:
        for vol_type in config.VOLATILITY_TYPES:
            for vol_window in config.VOLATILITY_WINDOWS:
                strategy_configs.append({
                    "momentum_type": "sma_slope",
                    "momentum_params": {"sma_window": sma_w, "slope_window": 5},
                    "volatility_type": vol_type,
                    "volatility_window": vol_window,
                    "name": f"SMASlope{sma_w}_{vol_type.upper()}{vol_window}"
                })
    
    print(f"Total strategy configurations: {len(strategy_configs)}")
    
    # Step 4: Run all strategies
    print("\nRunning TSMOM strategies...")
    strategy_results = {}
    strategy_metrics = {}
    full_results = {}  # Store full results for volatility comparison
    
    for idx, config_dict in enumerate(strategy_configs, 1):
        strategy_name = config_dict["name"]
        
        # Initialize strategy
        strategy = TSMOMStrategy(
            momentum_type=config_dict["momentum_type"],
            momentum_params=config_dict["momentum_params"],
            volatility_type=config_dict["volatility_type"],
            volatility_window=config_dict["volatility_window"],
            target_volatility=config.TARGET_VOLATILITY,
            commission_rate=config.COMMISSION_RATE
        )
        
        # Run backtest
        try:
            results = strategy.simulate_trading(trading_data)
            
            # Store returns
            strategy_results[strategy_name] = results["net_return"].dropna()
            full_results[strategy_name] = results
            
            # Calculate metrics
            metrics = PerformanceAnalyzer.calculate_metrics(results["net_return"].dropna())
            strategy_metrics[strategy_name] = metrics
            
            if idx % 10 == 0:
                print(f"  Completed {idx}/{len(strategy_configs)} strategies...")
                
        except Exception as e:
            print(f"  Error running {strategy_name}: {str(e)}")
            continue
    
    print(f"\nSuccessfully ran {len(strategy_results)} strategies")
    
    # Step 5: Calculate benchmark metrics
    print("\nCalculating benchmark metrics...")
    benchmark_metrics = PerformanceAnalyzer.calculate_metrics(
        benchmark_results["net_benchmark_return"].dropna()
    )
    strategy_metrics["Benchmark"] = benchmark_metrics
    
    print(f"  Benchmark | Sharpe: {benchmark_metrics['Sharpe Ratio']:.3f} | "
          f"Annual Return: {benchmark_metrics['Annual Return']*100:.2f}%")
    
    # Step 6: Display top performers
    print("\n" + "=" * 80)
    print("TOP 10 STRATEGIES BY SHARPE RATIO")
    print("=" * 80)
    
    metrics_df = pd.DataFrame(strategy_metrics).T
    top_10 = metrics_df.nlargest(10, "Sharpe Ratio")
    print("\n", top_10.to_string())
    
    # Step 7: Visualizations
    print("\nGenerating visualizations...")
    visualizer = StrategyVisualizer(config.PLOT_FIGSIZE, config.PLOT_DPI)
    
    # Plot 1: Cumulative returns (top strategies)
    visualizer.plot_cumulative_returns(
        strategy_results,
        benchmark_results["net_benchmark_return"].dropna(),
        title=f"TSMOM Strategies vs {config.BENCHMARK_TICKER} Benchmark",
        max_strategies=15
    )
    
    # Plot 2: Performance heatmap (top strategies)
    visualizer.plot_performance_heatmap(
        metrics_df,
        title="Performance Metrics Across TSMOM Configurations",
        top_n=20
    )
    
    # Plot 3: Compare volatility estimators (select a few strategies)
    print("\nComparing volatility estimators...")
    vol_comparison = {}
    for vol_type in config.VOLATILITY_TYPES:
        # Find a representative strategy for each volatility type
        for name, results in full_results.items():
            if vol_type.upper() in name:
                vol_comparison[f"{vol_type.upper()}"] = results
                break
    
    if len(vol_comparison) > 0:
        visualizer.plot_volatility_comparison(
            vol_comparison,
            title="Realized Volatility: Different Estimators Comparison"
        )
    
    print("\n" + "=" * 80)
    print("BACKTEST COMPLETE")
    print("=" * 80)
    
    # Summary statistics
    print("\nSummary Statistics:")
    print(f"  Total strategies tested: {len(strategy_results)}")
    print(f"  Strategies outperforming benchmark: "
          f"{sum(metrics_df['Sharpe Ratio'] > benchmark_metrics['Sharpe Ratio'])}")
    print(f"  Best Sharpe Ratio: {metrics_df['Sharpe Ratio'].max():.3f}")
    print(f"  Benchmark Sharpe Ratio: {benchmark_metrics['Sharpe Ratio']:.3f}")
    
    return strategy_results, strategy_metrics, benchmark_results, metrics_df


# ============================================================================
# RUN BACKTEST
# ============================================================================

if __name__ == "__main__":
    results, metrics, benchmark, metrics_df = main()